# CaseIQ: Phase 2 — LLM Fine-Tuning
### Fine-tuning Llama 2 7B with LoRA on 74 IFQM-aligned business case studies
**For: IFQM Bangalore & SRM Q Club | Submitted by Viva Baranwal**

> Before running: Upload `data/training_data.json` and `data/embeddings/` to a Google Drive folder called `CaseIQ/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json
import os

# We updated the path to include the /data/ folder!
file_path = '/content/drive/My Drive/CaseIQ/data/training_data.json'

if os.path.exists(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    print("🎉 Success! Colab found your file.")
    print(f"Total case templates available: {len(data)}")
else:
    print("❌ Still can't see it. Let me know if we need to troubleshoot further!")

!pip install -q transformers datasets peft bitsandbytes accelerate torch
!pip install -q sentence-transformers chromadb
print("All dependencies installed.")
import transformers
import datasets
import peft
import bitsandbytes
import accelerate
import torch

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
import os

for root, dirs, files in os.walk('/content/drive'):
    if 'training_data.json' in files:
        print(os.path.join(root, 'training_data.json'))

In [ ]:
import os

base_path = "/content/drive/MyDrive/CaseIQ"

print("Base path:", base_path)
print("Files:", os.listdir(base_path))

In [ ]:
import json

with open(f'{base_path}/data/training_data.json') as f:
    all_cases = json.load(f)

print(f"Total cases loaded: {len(all_cases)}")
print(f"Example company: {all_cases[0].get('company', all_cases[0].get('organization', 'N/A'))}")
print(f"Keys in each case: {list(all_cases[0].keys())}")

In [ ]:
training_messages = []

for case in all_cases:
    # Match your actual JSON keys
    company = case.get('title', 'Unknown Case')
    context = case.get('context', '')
    narrative = case.get('narrative_arc', '')
    themes = case.get('academic_themes', '')
    protagonist = case.get('protagonist', '')
    discussion = case.get('discussion_questions', '')
    stakeholders = case.get('stakeholders', '')

    # Skip incomplete cases
    if not context or not narrative:
        continue

    message = {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert academic case writer aligned with IFQM standards. Given company data, generate a structured teaching case study starting with an executive protagonist's dilemma. Sections: BACKGROUND, THEMES, INTERVENTION, RESULTS, LEARNING OUTCOMES. A case study describes what happened — never prescribe. Analysis belongs to the student."
            },
            {
                "role": "user",
                "content": f"Case Title: {company}\nProtagonist: {protagonist}\nStakeholders: {stakeholders}\nGenerate a complete IFQM-aligned case study."
            },
            {
                "role": "assistant",
                "content": f"BACKGROUND:\n{context}\n\nTHEMES:\n{themes}\n\nINTERVENTION:\n{narrative}\n\nRESULTS:\n{narrative}\n\nLEARNING OUTCOMES:\n{discussion}"
            }
        ]
    }
    training_messages.append(message)

print(f"Valid training examples: {len(training_messages)} out of {len(all_cases)} cases")

# Save formatted data to Drive
with open(f'{base_path}/training_formatted.json', 'w') as f:
    json.dump(training_messages, f)
print("Saved training_formatted.json to Drive.")

## Step 2: Download and Load Llama 2 7B
⏱️ This takes 10–15 minutes (downloading ~13 GB). Do not close the tab.

You need a Hugging Face account and must accept the Llama 2 license at:
https://huggingface.co/meta-llama/Llama-2-7b-hf

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-2-7b-hf"

from huggingface_hub import login
login(token="your_hf_token_here")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)
model = model.to("cuda")          # explicit move, no device_map="auto"

print(f"Model loaded: {model_name}")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
!pip uninstall -y torchao
!pip install -q --upgrade peft transformers accelerate bitsandbytes

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
import json

with open(f'{base_path}/training_formatted.json') as f:
    training_data = json.load(f)

def format_for_training(example):
    messages = example['messages']
    text = ""
    for msg in messages:
        if msg['role'] == 'system':
            text += f"<s>[INST] <<SYS>>\n{msg['content']}\n<</SYS>>\n\n"
        elif msg['role'] == 'user':
            text += f"{msg['content']} [/INST] "
        elif msg['role'] == 'assistant':
            text += f"{msg['content']} </s>"
    return {"text": text}

raw_dataset = Dataset.from_list(training_data)
dataset = raw_dataset.map(format_for_training)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names
)

split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")
print(f"Columns: {train_dataset.column_names}")

In [ ]:
print("all_cases:", 'all_cases' in dir())
print("training_messages:", 'training_messages' in dir())
print("training_data:", 'training_data' in dir())
print("raw_dataset:", 'raw_dataset' in dir())
print("dataset:", 'dataset' in dir())
print("model:", 'model' in dir())
print("tokenizer:", 'tokenizer' in dir())

## Step 3: Train the Model
⚠️ This cell takes 4–6 hours on a free Colab GPU (T4).

Do NOT close this tab. Loss should decrease over time — if it goes up, something is wrong.

Target final loss: **below 1.0**

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

training_args = TrainingArguments(
    output_dir=f"{base_path}/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=1,        # reduced from 2
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,        # increased to keep effective batch size = 8
    learning_rate=2e-4,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    logging_steps=25,
    max_grad_norm=1.0,
    remove_unused_columns=False,
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,          # NEW — trades speed for memory
    optim="adamw_bnb_8bit"                # NEW — 8-bit optimizer, less memory
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
save_path = f"{base_path}/final_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")
print(f"Files saved: {os.listdir(save_path)}")

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the saved model
test_tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/final_model")
test_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)
test_model = PeftModel.from_pretrained(test_model, f"{base_path}/final_model")

# Run a test prompt
prompt = """<s>[INST] <<SYS>>
You are an expert academic case writer aligned with IFQM standards.
<</SYS>>

Company: Wipro
Industry: IT Services
Generate a complete IFQM-aligned case study. [/INST]"""

inputs = test_tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

result = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated case study preview:")
print(result[len(prompt):])

## ✅ Training Complete

### Next Steps:
1. Download `final_model/` folder from Google Drive
2. Place it at: `project_live/model/final_model/`
3. Run the backend:
```bash
cd project_live/backend
uvicorn app:app --reload
```
4. Open `project_live/index.html` in browser
5. Fill the 9-screen wizard and generate your first real case study

### If Colab session times out mid-training:
- Checkpoints are saved every 100 steps to `CaseIQ/checkpoints/`
- Resume by loading the latest checkpoint in Cell 12:
  - Add `resume_from_checkpoint=True` to `trainer.train()`